In [1]:
import numpy as np
from pathlib import Path

# Dataset path

data_dir = Path("processed_data")
split_dir = data_dir / "split_dataset"

print("Split dataset directory:", split_dir.resolve())
print("Directory exists:", split_dir.exists())

Split dataset directory: E:\data of brain projec\data of brain project\EEG_project\processed_data\split_dataset
Directory exists: True


In [2]:
# Load split datasets

X_train = np.load(split_dir / "X_train.npy")
X_val = np.load(split_dir / "X_val.npy")
X_test = np.load(split_dir / "X_test.npy")

y_valence_train = np.load(split_dir / "y_valence_train.npy")
y_valence_val = np.load(split_dir / "y_valence_val.npy")
y_valence_test = np.load(split_dir / "y_valence_test.npy")

y_arousal_train = np.load(split_dir / "y_arousal_train.npy")
y_arousal_val = np.load(split_dir / "y_arousal_val.npy")
y_arousal_test = np.load(split_dir / "y_arousal_test.npy")

y_dominance_train = np.load(split_dir / "y_dominance_train.npy")
y_dominance_val = np.load(split_dir / "y_dominance_val.npy")
y_dominance_test = np.load(split_dir / "y_dominance_test.npy")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nValence:")
print(y_valence_train.shape, y_valence_val.shape, y_valence_test.shape)

print("\nArousal:")
print(y_arousal_train.shape, y_arousal_val.shape, y_arousal_test.shape)

print("\nDominance:")
print(y_dominance_train.shape, y_dominance_val.shape, y_dominance_test.shape)

X_train: (59360, 224)
X_val: (11130, 224)
X_test: (14840, 224)

Valence:
(59360,) (11130,) (14840,)

Arousal:
(59360,) (11130,) (14840,)

Dominance:
(59360,) (11130,) (14840,)


In [3]:
# Check target values

targets = {
    "Valence": [
        y_valence_train,
        y_valence_val,
        y_valence_test
    ],
    "Arousal": [
        y_arousal_train,
        y_arousal_val,
        y_arousal_test
    ],
    "Dominance": [
        y_dominance_train,
        y_dominance_val,
        y_dominance_test
    ]
}

for name, splits in targets.items():

    print(f"\n{name}")

    for split_name, y in zip(
        ["Train", "Validation", "Test"],
        splits
    ):
        print(
            f"{split_name}:",
            "dtype =", y.dtype,
            "min =", y.min(),
            "max =", y.max(),
            "unique =", np.unique(y)
        )


Valence
Train: dtype = float32 min = 1.0 max = 5.0 unique = [1. 2. 3. 4. 5.]
Validation: dtype = float32 min = 1.0 max = 5.0 unique = [1. 2. 3. 4. 5.]
Test: dtype = float32 min = 1.0 max = 5.0 unique = [1. 2. 3. 4. 5.]

Arousal
Train: dtype = float32 min = 1.0 max = 5.0 unique = [1. 2. 3. 4. 5.]
Validation: dtype = float32 min = 2.0 max = 5.0 unique = [2. 3. 4. 5.]
Test: dtype = float32 min = 1.0 max = 5.0 unique = [1. 2. 3. 4. 5.]

Dominance
Train: dtype = float32 min = 1.0 max = 5.0 unique = [1. 2. 3. 4. 5.]
Validation: dtype = float32 min = 2.0 max = 5.0 unique = [2. 3. 4. 5.]
Test: dtype = float32 min = 1.0 max = 5.0 unique = [1. 2. 3. 4. 5.]


In [5]:
from sklearn.ensemble import RandomForestClassifier
import joblib

rf_valence = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest for Valence...")

rf_valence.fit(
    X_train,
    y_valence_train
)

print("Training completed.")

Training Random Forest for Valence...
Training completed.


In [6]:
# Load subject information

subjects_train = np.load(
    split_dir / "subjects_train.npy"
)

subjects_val = np.load(
    split_dir / "subjects_val.npy"
)

subjects_test = np.load(
    split_dir / "subjects_test.npy"
)

print("Subjects train:", subjects_train.shape)
print("Subjects val:", subjects_val.shape)
print("Subjects test:", subjects_test.shape)

print("Unique train subjects:", np.unique(subjects_train))

Subjects train: (59360,)
Subjects val: (11130,)
Subjects test: (14840,)
Unique train subjects: [ 3  4  5  6  7  8 11 12 14 15 17 19 20 21 22 23]


In [7]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(
    n_splits=5
)

print("Number of folds:", group_kfold.n_splits)

Number of folds: 5


In [8]:
# Check subject separation in each fold

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X_train, y_valence_train, groups=subjects_train),
    start=1
):
    train_subjects_fold = np.unique(subjects_train[train_idx])
    val_subjects_fold = np.unique(subjects_train[val_idx])

    overlap = np.intersect1d(
        train_subjects_fold,
        val_subjects_fold
    )

    print(f"Fold {fold}")
    print("Train subjects:", train_subjects_fold)
    print("Validation subjects:", val_subjects_fold)
    print("Overlap:", overlap)
    print()

Fold 1
Train subjects: [ 4  5  6  7 11 12 14 15 19 20 21 22]
Validation subjects: [ 3  8 17 23]
Overlap: []

Fold 2
Train subjects: [ 3  4  5  6  8 11 12 14 17 19 20 21 23]
Validation subjects: [ 7 15 22]
Overlap: []

Fold 3
Train subjects: [ 3  4  5  7  8 11 12 15 17 19 20 22 23]
Validation subjects: [ 6 14 21]
Overlap: []

Fold 4
Train subjects: [ 3  4  6  7  8 11 14 15 17 19 21 22 23]
Validation subjects: [ 5 12 20]
Overlap: []

Fold 5
Train subjects: [ 3  5  6  7  8 12 14 15 17 20 21 22 23]
Validation subjects: [ 4 11 19]
Overlap: []



In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

cv_scores = []

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(
        X_train,
        y_valence_train,
        groups=subjects_train
    ),
    start=1
):
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[train_idx],
        y_valence_train[train_idx]
    )

    predictions = model.predict(
        X_train[val_idx]
    )

    accuracy = accuracy_score(
        y_valence_train[val_idx],
        predictions
    )

    cv_scores.append(accuracy)

    print(f"Fold {fold} Accuracy: {accuracy:.4f}")

print("\nMean CV Accuracy:", np.mean(cv_scores))
print("Std CV Accuracy:", np.std(cv_scores))

Fold 1 Accuracy: 0.2211
Fold 2 Accuracy: 0.2527
Fold 3 Accuracy: 0.1727
Fold 4 Accuracy: 0.1583
Fold 5 Accuracy: 0.2060

Mean CV Accuracy: 0.2021698113207547
Std CV Accuracy: 0.03383227045969199


In [10]:
from sklearn.metrics import confusion_matrix, classification_report

all_true = []
all_pred = []

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(
        X_train,
        y_valence_train,
        groups=subjects_train
    ),
    start=1
):
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[train_idx],
        y_valence_train[train_idx]
    )

    predictions = model.predict(
        X_train[val_idx]
    )

    all_true.extend(y_valence_train[val_idx])
    all_pred.extend(predictions)

print("Classification Report:")
print(
    classification_report(
        all_true,
        all_pred,
        labels=[1, 2, 3, 4, 5],
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        all_true,
        all_pred,
        labels=[1, 2, 3, 4, 5]
    )
)

Classification Report:
              precision    recall  f1-score   support

           1     0.1694    0.1881    0.1782     11848
           2     0.1441    0.0926    0.1128     11088
           3     0.2249    0.2984    0.2565     13068
           4     0.2562    0.2945    0.2740     13905
           5     0.1428    0.0869    0.1080      9451

    accuracy                         0.2034     59360
   macro avg     0.1875    0.1921    0.1859     59360
weighted avg     0.1930    0.2034    0.1945     59360

Confusion Matrix:
[[2229 1622 3936 3118  943]
 [2618 1027 3318 2802 1323]
 [3442 1319 3899 3156 1252]
 [3266 1518 3617 4095 1409]
 [1607 1642 2567 2814  821]]


In [11]:
from sklearn.metrics import accuracy_score, classification_report

rf_valence_final = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest on full training set...")

rf_valence_final.fit(
    X_train,
    y_valence_train
)

print("Training completed.")

valence_val_pred = rf_valence_final.predict(X_val)

valence_val_accuracy = accuracy_score(
    y_valence_val,
    valence_val_pred
)

print(f"Validation Accuracy: {valence_val_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_valence_val,
        valence_val_pred,
        labels=[1, 2, 3, 4, 5],
        digits=4
    )
)

Training Random Forest on full training set...
Training completed.
Validation Accuracy: 0.2439

Classification Report:
              precision    recall  f1-score   support

           1     0.2616    0.5915    0.3627      1939
           2     0.3540    0.2712    0.3071      3035
           3     0.0938    0.1416    0.1129      1963
           4     0.3282    0.1740    0.2275      2574
           5     0.2065    0.0117    0.0222      1619

    accuracy                         0.2439     11130
   macro avg     0.2488    0.2380    0.2065     11130
weighted avg     0.2646    0.2439    0.2227     11130



In [12]:
# Random Forest hyperparameter candidates

rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

print("Hyperparameter candidates:")
for parameter, values in rf_params.items():
    print(f"{parameter}: {values}")

Hyperparameter candidates:
n_estimators: [100, 200, 300]
max_depth: [None, 10, 20]
min_samples_split: [2, 5]
min_samples_leaf: [1, 2]
max_features: ['sqrt', 'log2']


In [13]:
from sklearn.model_selection import RandomizedSearchCV

rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_params,
    n_iter=15,
    scoring="accuracy",
    cv=group_kfold,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("RandomizedSearchCV is ready.")

RandomizedSearchCV is ready.


In [14]:
# Run RandomizedSearchCV

print("Starting Random Forest hyperparameter search...")

rf_random_search.fit(
    X_train,
    y_valence_train,
    groups=subjects_train
)

print("Hyperparameter search completed.")

Starting Random Forest hyperparameter search...
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Hyperparameter search completed.


In [15]:
print("Best CV Accuracy:", rf_random_search.best_score_)
print("Best Parameters:")
print(rf_random_search.best_params_)

Best CV Accuracy: 0.2035893980233603
Best Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}


In [16]:
rf_valence_tuned = rf_random_search.best_estimator_

valence_val_pred_tuned = rf_valence_tuned.predict(X_val)

valence_val_accuracy_tuned = accuracy_score(
    y_valence_val,
    valence_val_pred_tuned
)

print(f"Validation Accuracy: {valence_val_accuracy_tuned:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_valence_val,
        valence_val_pred_tuned,
        labels=[1, 2, 3, 4, 5],
        digits=4
    )
)

Validation Accuracy: 0.2401

Classification Report:
              precision    recall  f1-score   support

           1     0.2650    0.5745    0.3627      1939
           2     0.3392    0.2521    0.2892      3035
           3     0.0879    0.1375    0.1073      1963
           4     0.3340    0.1977    0.2484      2574
           5     0.1818    0.0086    0.0165      1619

    accuracy                         0.2401     11130
   macro avg     0.2416    0.2341    0.2048     11130
weighted avg     0.2579    0.2401    0.2208     11130



In [17]:
# Save tuned Random Forest model

rf_valence_path = models_dir / "random_forest_valence_tuned.joblib"

joblib.dump(
    rf_valence_tuned,
    rf_valence_path
)

print("Random Forest model saved.")
print("Path:", rf_valence_path.resolve())

Random Forest model saved.
Path: E:\data of brain projec\data of brain project\EEG_project\models\random_forest_valence_tuned.joblib
